# College Basketball Playstyle Clustering
A Data Science Case Study in Python

## 1. Project Overview
This project analyzes college basketball teams (2013-2024) by grouping them into playstyle clusters using advanced statistics from Torvik.
The goal is to identify which playstyles are most successful in the NCAA Tournament and apply those insights to evaluate current teams.


## 2. Data Loading and Preparation

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv("cbb.csv")


In [ ]:
features = [
   "ADJ_T",
    "EFG_O","EFG_D",
    "TOR","TORD",
    "ORB","DRB",
    "FTR","FTRD",
    "2P_O","2P_D",
    "3P_O","3P_D"
]

In [ ]:
X = df[features].dropna()

df_model = df.loc[X.index].copy()

## 3. Clustering Playstyles

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
df_model["cluster"] = kmeans.fit_predict(X_scaled)

## 5. Playstyle Visualization (PCA Map)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)

pcs = pca.fit_transform(X_scaled)

df_model["PC1"] = pcs[:,0]
df_model["PC2"] = pcs[:,1]

In [ ]:
plt.figure(figsize=(13, 9))

sns.scatterplot(
    data=df_model,
    x="PC1",
    y="PC2",
    hue="cluster",
    palette="tab10",
    alpha=0.35,
    s=40,
    legend="full"
)

plt.title("College Basketball Playstyle Map (2013–2024)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.show()

This chart shows how teams group into different playstyles based on their statistical profiles.

## 6. Understanding Cluster Playstyles

In [ ]:
cluster_profile = df_model.groupby("cluster")[[
    "ADJOE","ADJDE","BARTHAG",
    "EFG_O","EFG_D",
    "TOR","TORD",
    "ORB","DRB",
    "FTR","FTRD",
    "2P_O","2P_D",
    "3P_O","3P_D",
    "ADJ_T","WAB"
]].mean().round(3)

cluster_profile

## 7. Tournament Success by Playstyle

In [ ]:
tournament_df = df_model[df_model["POSTSEASON"] != "Missed"]

In [ ]:
important_stats = [
    "ADJ_T",
    "EFG_O","EFG_D",
    "TOR","TORD",
    "ORB",
    "3P_O"
]

cluster_profile_tourney = tournament_df.groupby("cluster")[important_stats].mean()

cluster_profile_tourney

In [ ]:
from sklearn.preprocessing import StandardScaler

important_stats = [
    "ADJ_T",
    "EFG_O","EFG_D",
    "TOR","TORD",
    "ORB",
    "3P_O"
]

cluster_profile = tournament_df.groupby("cluster")[important_stats].mean()

profile_scaler = StandardScaler()
cluster_scaled = profile_scaler.fit_transform(cluster_profile)

cluster_scaled = pd.DataFrame(
    cluster_scaled,
    index=cluster_profile.index,
    columns=cluster_profile.columns
)

ax = cluster_scaled.plot(kind="bar", figsize=(18,6))

plt.title("Relative Playstyle Differences by Cluster (Bars = Z-score, Labels = Actual Stat Value)")
plt.ylabel("Standardized Score (Z-score)")
plt.xticks(rotation=0)

plt.axhline(0, color="black", linewidth=1)

# stats where LOWER is better
lower_is_better = ["EFG_D", "TOR"]

for stat_i, stat in enumerate(cluster_scaled.columns):

    if stat in lower_is_better:
        best_cluster = cluster_profile[stat].idxmin()
    else:
        best_cluster = cluster_profile[stat].idxmax()

    real_value = cluster_profile.loc[best_cluster, stat]

    container = ax.containers[stat_i]
    bar = container[best_cluster]

    x = bar.get_x() + bar.get_width() / 2
    y = bar.get_height()

    x_offset = 0.35 if stat_i % 2 == 0 else -0.35
    y_offset = 0.35 if y >= 0 else -0.35

    ax.annotate(
        f"{real_value:.2f}",
        xy=(x, y),
        xytext=(x + x_offset, y + y_offset),
        ha="center",
        arrowprops=dict(arrowstyle="->", lw=1.2),
        fontsize=10,
        fontweight="bold"
    )

plt.show()

## Interpreting Playstyle Clusters (Tournament Teams)

The clustering algorithm grouped teams based on several statistical indicators that describe how teams play, including tempo, shooting efficiency, turnover rate, defensive ball pressure, rebounding, and three-point shooting. Each cluster represents a group of teams with similar statistical playstyles.

The bar chart above displays standardized (Z-score) values for the key playstyle statistics. Positive values indicate that the cluster performs above the overall average for tournament teams, while negative values indicate below-average performance.

### Cluster 0 – Slow Defensive Teams
Teams in Cluster 0 tend to play at a slower tempo and rely heavily on defensive efficiency. They allow a lower opponent effective field goal percentage than most other clusters, indicating strong defensive performance. These teams generally control the pace of the game and focus on limiting opponent scoring rather than relying on fast offensive production.

### Cluster 1 – Balanced Teams
Cluster 1 teams appear relatively balanced across most statistical categories. They do not strongly dominate any single playstyle metric but perform near the average in most areas such as tempo, shooting efficiency, turnover rate, and rebounding. These teams tend to rely on a balanced overall approach rather than one extreme style.

### Cluster 2 – Physical Rebounding Teams
Teams in Cluster 2 emphasize rebounding and physical interior play. They show the highest offensive rebounding rates among the clusters but lower shooting efficiency and slower offensive tempo. These teams often rely on second-chance opportunities and physical play near the basket to generate scoring.

### Cluster 3 – Efficient Two-Way Teams
Cluster 3 teams perform well in both offensive and defensive efficiency. They show strong effective field goal percentages on offense while also maintaining solid defensive shooting numbers. These teams tend to combine efficient offense with consistent overall play.

### Cluster 4 – Turnover-Forcing Teams
Teams in Cluster 4 stand out for their ability to pressure opponents and create mistakes. They perform well in defensive turnover rate, which suggests a playstyle built around disrupting opposing offenses. These teams often rely on defensive aggression to create extra possessions and scoring chances.

### Cluster 5 – Elite Shooting Teams
Cluster 5 teams stand out for their offensive shooting ability. They have the highest offensive effective field goal percentage and strong three-point shooting rates. These teams rely heavily on perimeter shooting and efficient offensive spacing to generate scoring opportunities.

### Summary
Overall, the clustering reveals several distinct tournament playstyles, ranging from defensive-focused teams to rebounding-heavy teams and elite shooting offenses. Later sections of this analysis examine how these playstyles relate to tournament success and advancement through the NCAA Tournament.

In [ ]:
round_table = pd.crosstab(df_model["cluster"], df_model["POSTSEASON"])

round_table

In [ ]:
round_order = [
    "R68",
    "R64",
    "R32",
    "S16",
    "E8",
    "F4",
    "2ND",
    "Champions"
]

round_table = round_table.reindex(columns=round_order)

round_table = round_table.fillna(0)

round_table

In [ ]:
round_table["Total Teams"] = round_table.sum(axis=1)

round_table

### Postseason Note

The `POSTSEASON` column shows the **furthest round each team reached** in that season’s NCAA Tournament.  
Each team appears only once, based on its final finish (`R68`, `R64`, `R32`, `S16`, `E8`, `F4`, `2ND`, or `Champions`).

Because of that, the round table shows **where teams finished**, not every round they played.

In [ ]:
success_rates = round_table.copy()

rounds = ["R68","R64","R32","S16","E8","F4","2ND","Champions"]

for r in rounds:
    success_rates[r + "_share"] = (
        success_rates[r] / success_rates[r].sum() * 100
    ).round(1)

success_rates

In [ ]:
success_summary = success_rates[[
    "Total Teams",
    "R64_share",
    "R32_share",
    "S16_share",
    "E8_share",
    "F4_share",
    "2ND_share",
    "Champions_share"
]]

success_summary

In [ ]:
success_summary.style.format({
    "Total Teams": "{:.0f}",
    "R64_share": "{:.1f}%",
    "R32_share": "{:.1f}%",
    "S16_share": "{:.1f}%",
    "E8_share": "{:.1f}%",
    "F4_share": "{:.1f}%",
    "2ND_share": "{:.1f}%",
    "Champions_share": "{:.1f}%"
})



In [ ]:
cluster_sizes = df_model["cluster"].value_counts().sort_index()

cluster_success_rate = pd.DataFrame({
    "Teams_in_cluster": cluster_sizes,
    "Sweet_16": round_table["S16"],
    "Elite_8": round_table["E8"],
    "Final_Four": round_table["F4"],
    "Champions": round_table["Champions"]
})

cluster_success_rate["Sweet16_rate"] = (
    cluster_success_rate["Sweet_16"] /
    cluster_success_rate["Teams_in_cluster"] * 100
).round(2)

cluster_success_rate["Elite8_rate"] = (
    cluster_success_rate["Elite_8"] /
    cluster_success_rate["Teams_in_cluster"] * 100
).round(2)

cluster_success_rate["Final4_rate"] = (
    cluster_success_rate["Final_Four"] /
    cluster_success_rate["Teams_in_cluster"] * 100
).round(2)

cluster_success_rate["Champion_rate"] = (
    cluster_success_rate["Champions"] /
    cluster_success_rate["Teams_in_cluster"] * 100
).round(2)

cluster_success_rate = cluster_success_rate[[
    "Teams_in_cluster",
    "Sweet_16", "Sweet16_rate",
    "Elite_8", "Elite8_rate",
    "Final_Four", "Final4_rate",
    "Champions", "Champion_rate"
]]

cluster_success_rate

### Tournament Success Rates Within Each Cluster

The tables above show how much each cluster contributes to different tournament rounds, such as the Sweet 16, Final Four, and national champions. However, those tables do not account for the fact that some clusters contain more teams than others.

To address that, the table below calculates success rates within each cluster. This shows the percentage of teams in each cluster that reached major tournament milestones, including the Sweet 16, Elite Eight, Final Four, and national championship.

This view is useful because it measures how successful each playstyle has been relative to its size. In other words, it helps answer the question: if a team belongs to a certain playstyle cluster, how often does that type of team make a deep NCAA Tournament run?

## 8. Applying Model to 2026 Teams

In [ ]:
torvik_raw = pd.read_excel("torvik_2026.xlsx")

# Fix the shifted columns caused by the pasted sheet format
torvik_df = torvik_raw.rename(columns={
    "Conf": "TEAM",
    "G": "CONF",
    "Rec": "G",
    "AdjOE": "Rec",
    "AdjDE": "ADJOE",
    "Barthag": "ADJDE",
    "EFG%": "BARTHAG",
    "EFGD%": "EFG_O",
    "TOR": "EFG_D",
    "TORD": "TOR",
    "ORB": "TORD",
    "DRB": "ORB",
    "FTR": "DRB",
    "FTRD": "FTR",
    "2P%": "FTRD",
    "2P%D": "2P_O",
    "3P%": "2P_D",
    "3P%D": "3P_O",
    "3PR": "3P_D",
    "3PRD": "3PR",
    "Adj T.": "3PRD",
    "WAB": "ADJ_T"
}).copy()

# Keep only real team rows
torvik_df = torvik_df[torvik_df["TEAM"].notna()].copy()
torvik_df = torvik_df[torvik_df["TEAM"] != "Team"].copy()
torvik_df = torvik_df[~torvik_df["TEAM"].astype(str).str.contains("seed", case=False, na=False)].copy()
torvik_df = torvik_df.reset_index(drop=True)

# Convert model features to numeric
for col in features:
    torvik_df[col] = pd.to_numeric(torvik_df[col], errors="coerce")

print("Rows after cleaning:", torvik_df.shape[0])
torvik_df[["TEAM", "CONF"] + features[:5]].head()

Stats as of 3/15/2026 8:52 pm cst

In [ ]:
current_X = torvik_df[features].dropna()
print("Teams surviving model features:", current_X.shape[0])
current_X.head()

In [ ]:
torvik_clean = torvik_df.loc[current_X.index].copy()
print("torvik_clean rows:", torvik_clean.shape[0])

In [ ]:
current_scaled = scaler.transform(current_X)

In [ ]:
torvik_clean["cluster"] = kmeans.predict(current_scaled)
torvik_clean[["TEAM", "CONF", "cluster"]].head(15)

In [ ]:
current_pcs = pca.transform(current_scaled)
torvik_clean["PC1"] = current_pcs[:, 0]
torvik_clean["PC2"] = current_pcs[:, 1]

torvik_clean[["TEAM", "CONF", "cluster", "PC1", "PC2"]].head(15)

In [ ]:
torvik_results = torvik_clean.merge(
    success_summary,
    left_on="cluster",
    right_index=True,
    how="left"
)

torvik_results.head(15)

In [ ]:
actual_tourney_teams = [
    # EAST
    "Duke", "Siena", "Ohio St.", "TCU", "St. John's", "Northern Iowa",
    "Kansas", "Cal Baptist", "Louisville", "South Florida", "Michigan St.",
    "North Dakota St.", "UCLA", "UCF", "UConn", "Connecticut", "Furman",

    # SOUTH
    "Florida", "Lehigh", "Prairie View A&M", "PVAMU", "Clemson", "Iowa",
    "Vanderbilt", "McNeese", "McNeese St.", "Nebraska", "Troy",
    "North Carolina", "VCU", "Illinois", "Penn", "Saint Mary's",
    "Texas A&M", "Houston", "Idaho",

    # WEST
    "Arizona", "Long Island", "LIU", "Villanova", "Utah St.", "Wisconsin",
    "High Point", "Arkansas", "Hawaii", "BYU", "NC State", "N.C. State",
    "Texas", "Gonzaga", "Kennesaw St.", "Miami FL", "Missouri", "Purdue",
    "Queens", "Queens (N.C.)",

    # MIDWEST
    "Michigan", "Howard", "UMBC", "Georgia", "Saint Louis", "Texas Tech",
    "Akron", "Alabama", "Hofstra", "Tennessee", "SMU", "Miami OH",
    "Virginia", "Wright St.", "Kentucky", "Santa Clara", "Iowa St.",
    "Tennessee St."
]

torvik_plot = torvik_results[
    torvik_results["TEAM"].isin(actual_tourney_teams)
].copy()

print("Teams found:", torvik_plot.shape[0])
print(sorted(torvik_plot["TEAM"].unique()))

In [ ]:
torvik_df.columns

In [ ]:
plt.figure(figsize=(13,9))
ax = plt.gca()


palette = sns.color_palette("tab10", 6)

# Historical teams (background)
sns.scatterplot(
    data=df_model,
    x="PC1",
    y="PC2",
    hue="cluster",
    palette=palette,
    alpha=0.12,
    legend=False,
    s=40
)

# 2026 teams (foreground)
sns.scatterplot(
    data=torvik_plot,
    x="PC1",
    y="PC2",
    hue="cluster",
    palette=palette,
    s=220,
    edgecolor="black",
    linewidth=1.0,
    alpha=0.95,
    legend=False
)




plt.title("College Basketball Playstyle Map (Historical Teams + 2026 Field)")
plt.xlabel("PC1")
plt.ylabel("PC2")

plt.show()

In [ ]:
torvik_plot["cluster"].value_counts().sort_index()

In [ ]:
torvik_plot[["TEAM", "CONF", "cluster", "Champions_share", "F4_share"]].sort_values(
    ["cluster", "Champions_share"], ascending=[True, False]
)

## 9. Identifying Championship Profiles

In [ ]:
champions_df = df_model[df_model["POSTSEASON"] == "Champions"].copy()

champions_df[["TEAM","YEAR","PC1","PC2"]]

In [ ]:
champion_center = champions_df[["PC1","PC2"]].mean()

champion_center

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

pc_scaler = StandardScaler()

champion_pcs_scaled = pc_scaler.fit_transform(champions_df[["PC1", "PC2"]])
champion_center_scaled = champion_pcs_scaled.mean(axis=0)

current_pcs_scaled = pc_scaler.transform(torvik_plot[["PC1", "PC2"]])

torvik_plot["champion_distance"] = np.sqrt(
    (current_pcs_scaled[:, 0] - champion_center_scaled[0])**2 +
    (current_pcs_scaled[:, 1] - champion_center_scaled[1])**2
)

In [ ]:
champion_similarity = torvik_plot.sort_values("champion_distance")

champion_similarity[
    ["TEAM","CONF","cluster","champion_distance"]
].head(20)

In [ ]:
champion_contenders = champion_similarity[
    champion_similarity["BARTHAG"] >= 0.900
]

champion_contenders.head(15)

In [ ]:
# smaller distance = better, so convert to similarity
torvik_plot["champion_similarity"] = 1 / (1 + torvik_plot["champion_distance"])

# optional normalized version
torvik_plot["champion_similarity_norm"] = (
    (torvik_plot["champion_similarity"] - torvik_plot["champion_similarity"].min()) /
    (torvik_plot["champion_similarity"].max() - torvik_plot["champion_similarity"].min())
)

In [ ]:
torvik_plot["cluster_strength"] = (
    torvik_plot["Champions_share"] * 0.7 +
    torvik_plot["F4_share"] * 0.3
)

In [ ]:
def minmax(series):
    return (series - series.min()) / (series.max() - series.min())

torvik_plot["BARTHAG_norm"] = minmax(torvik_plot["BARTHAG"])
torvik_plot["cluster_strength_norm"] = minmax(torvik_plot["cluster_strength"])

In [ ]:
torvik_plot["Champion_Score"] = (
    torvik_plot["champion_similarity_norm"] * 0.45 +
    torvik_plot["BARTHAG_norm"] * 0.40 +
    torvik_plot["cluster_strength_norm"] * 0.15
)

champion_ranking = torvik_plot.sort_values("Champion_Score", ascending=False).copy()

champion_ranking[
    ["TEAM", "CONF", "cluster", "BARTHAG", "champion_distance", "Champion_Score"]
].head(25)

In [ ]:
plt.figure(figsize=(13,9))
ax = plt.gca()

sns.scatterplot(
    data=torvik_plot,
    x="champion_similarity_norm",
    y="BARTHAG_norm",
    hue="cluster",
    size="Champion_Score",
    sizes=(80, 700),
    palette="tab10",
    edgecolor="black",
    alpha=0.9
)

# label top teams
top_labels = torvik_plot.sort_values("Champion_Score", ascending=False).head(20)

for _, row in top_labels.iterrows():
    ax.text(
        row["champion_similarity_norm"] + 0.01,
        row["BARTHAG_norm"] + 0.005,
        row["TEAM"],
        fontsize=9
    )

plt.axvline(0.6, linestyle="--", color="gray", alpha=0.7)
plt.axhline(0.6, linestyle="--", color="gray", alpha=0.7)

plt.title("2026 Champion Profile Map")
plt.xlabel("Champion Playstyle Similarity")
plt.ylabel("Stronger Resume (BARTHAG)")
plt.xlim(-0.02, 1.05)
plt.ylim(-0.02, 1.05)
plt.grid(alpha=0.2)

plt.show()

### Champion Profile Map (2026 Teams)

This chart compares current 2026 teams to the statistical profile of historical NCAA champions.

Each team is evaluated using three key indicators:

- **Champion Similarity (x-axis)** — how similar the team’s playstyle is to historical champions based on PCA distance.
- **BARTHAG (y-axis)** — an overall team strength metric based on adjusted offensive and defensive efficiency.
- **Champion Score (circle size)** — a combined metric using champion similarity, team strength, and the historical success of the team’s playstyle cluster.

Teams appearing toward the upper-right corner of the chart are both:

- statistically similar to past championship teams
- and strong overall teams based on efficiency metrics.

These teams represent the strongest championship candidates entering the NCAA Tournament.

In [ ]:
top_n = 20
leaderboard = champion_ranking.head(top_n).sort_values("Champion_Score", ascending=True)

plt.figure(figsize=(11,9))
ax = plt.gca()

sns.barplot(
    data=leaderboard,
    x="Champion_Score",
    y="TEAM",
    hue="cluster",
    dodge=False,
    palette="tab10"
)

# add score labels
for i, row in leaderboard.iterrows():
    ax.text(
        row["Champion_Score"] + 0.005,
        leaderboard.index.get_loc(i),
        f"{row['Champion_Score']:.2f}",
        va="center"
    )

plt.title("2026 Champion Score Leaderboard")
plt.xlabel("Champion Score")
plt.ylabel("")
plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
champion_ranking[
    ["TEAM","BARTHAG","champion_similarity_norm","cluster_strength_norm","Champion_Score"]
].head(15)

In [ ]:
df_model["is_champion"] = df_model["POSTSEASON"] == "Champions"
df_model["is_final4"] = df_model["POSTSEASON"].isin(["Champions", "2ND", "F4"])

In [ ]:
for cluster_id in sorted(df_model["cluster"].unique()):

    hist_cluster = df_model[df_model["cluster"] == cluster_id].copy()
    current_cluster = torvik_plot[torvik_plot["cluster"] == cluster_id].copy()

    plt.figure(figsize=(12, 8))
    ax = plt.gca()

    # all historical teams in cluster
    plt.scatter(
        hist_cluster["PC1"],
        hist_cluster["PC2"],
        s=30,
        alpha=0.12,
        color="gray"
    )

    # historical Final Four teams
    hist_f4 = hist_cluster[hist_cluster["POSTSEASON"].isin(["Champions", "2ND", "F4"])]
    plt.scatter(
        hist_f4["PC1"],
        hist_f4["PC2"],
        s=80,
        alpha=0.5,
        color="orange",
        label="Historical Final Four"
    )

    # historical champions
    hist_champs = hist_cluster[hist_cluster["POSTSEASON"] == "Champions"]
    plt.scatter(
        hist_champs["PC1"],
        hist_champs["PC2"],
        s=200,
        color="gold",
        edgecolor="black",
        marker="*",
        label="Historical Champions"
    )

    # make sure BARTHAG is numeric before using it for point sizes
    current_cluster["BARTHAG"] = pd.to_numeric(current_cluster["BARTHAG"], errors="coerce")
    point_sizes = (120 + current_cluster["BARTHAG"].fillna(0).clip(lower=0) * 12).to_numpy()

    # current teams
    if len(current_cluster) > 0:
        plt.scatter(
            current_cluster["PC1"],
            current_cluster["PC2"],
            s=point_sizes,
            color="red",
            edgecolor="black",
            alpha=0.9,
            label="2026 Teams"
        )

        for _, row in current_cluster.iterrows():
            ax.text(
                row["PC1"] + 0.03,
                row["PC2"] + 0.03,
                row["TEAM"],
                fontsize=8
            )

    plt.title(f"Cluster {cluster_id}: 2026 Teams vs Historical Teams")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

### Cluster-by-Cluster Comparison: 2026 Teams vs Historical Tournament Teams

The charts below compare current 2026 teams to historical teams within each playstyle cluster. Each plot isolates one cluster and shows how current teams fit within that historical playstyle group.

In each chart:

- **Gray points** represent all historical teams in that cluster  
- **Orange points** represent historical Final Four teams  
- **Gold stars** represent historical national champions  
- **Red points** represent current 2026 teams, with point size scaled by **BARTHAG (overall team strength)**

These charts help show whether current teams within each playstyle cluster resemble the historical teams that have made deep NCAA Tournament runs or won national championships.

## 10. Key Insights and Takeaways

This project shows that certain playstyles are more likely to produce deep tournament runs. Teams that combine strong efficiency metrics with historically successful playstyles tend to have the highest championship potential.

In [ ]:
torvik_plot.to_csv("torvik_playstyle_results_2026.csv", index=False)
df_model.to_csv("historical_playstyle_results.csv", index=False)
champion_ranking.to_csv("champion_ranking_2026.csv", index=False)